In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import cv2
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
class AgeGenderCNN(nn.Module):
    def __init__(self, input_size=(128, 128)):
        super(AgeGenderCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2)
        )

        # Flatten sonrası boyutu otomatik hesapla
        dummy_input = torch.zeros(1, 1, *input_size)
        dummy_output = self.features(dummy_input)
        self.flattened_size = dummy_output.view(1, -1).shape[1]

        # Cinsiyet tahmini başlığı
        self.gender_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

        # Yaş tahmini başlığı
        self.age_head = nn.Sequential(
            nn.Linear(self.flattened_size, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        gender_out = torch.sigmoid(self.gender_head(x))
        age_out = self.age_head(x)
        return gender_out, age_out


In [3]:
class AgeGenderDataset(Dataset):
    def __init__(self, X, y_gender, y_age):
        self.X = X.astype(np.float32) / 255.0
        self.y_gender = y_gender.astype(np.float32).reshape(-1, 1)
        self.y_age = y_age.astype(np.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        x = np.expand_dims(x, axis=0)
        return torch.tensor(x), torch.tensor(self.y_gender[idx]), torch.tensor(self.y_age[idx])

In [4]:
def train_model(model, dataloader, criterion_g, criterion_a, optimizer, device):
    model.train()
    total_loss = 0
    for x, y_g, y_a in tqdm(dataloader):
        x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
        optimizer.zero_grad()
        pred_g, pred_a = model(x)
        loss_g = criterion_g(pred_g, y_g)
        loss_a = criterion_a(pred_a, y_a)
        loss = loss_g + loss_a
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

In [5]:
def evaluate_model(model, dataloader, criterion_g, criterion_a, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x, y_g, y_a = x.to(device), y_g.to(device), y_a.to(device)
            pred_g, pred_a = model(x)
            loss_g = criterion_g(pred_g, y_g)
            loss_a = criterion_a(pred_a, y_a)
            loss = loss_g + loss_a
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [6]:
def test_model(model, dataloader, device):
    model.eval()
    all_preds_g = []
    all_preds_a = []
    all_true_g = []
    all_true_a = []

    with torch.no_grad():
        for x, y_g, y_a in dataloader:
            x = x.to(device)
            pred_g, pred_a = model(x)
            all_preds_g += pred_g.cpu().numpy().flatten().tolist()
            all_preds_a += pred_a.cpu().numpy().flatten().tolist()
            all_true_g += y_g.numpy().flatten().tolist()
            all_true_a += y_a.numpy().flatten().tolist()

    pred_g_bin = [1 if p > 0.5 else 0 for p in all_preds_g]

    print("\n=== [TEST SONUÇLARI] ===")
    print("Cinsiyet - Accuracy :", accuracy_score(all_true_g, pred_g_bin))
    print("Cinsiyet - Precision:", precision_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - Recall   :", recall_score(all_true_g, pred_g_bin, zero_division=0))
    print("Cinsiyet - F1-score :", f1_score(all_true_g, pred_g_bin, zero_division=0))

    print("Yaş - MAE           :", mean_absolute_error(all_true_a, all_preds_a))
    print("Yaş - RMSE          :", root_mean_squared_error(all_true_a, all_preds_a))

In [7]:
X_train = np.load("X_train_utkface.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_utkface.npy")
y_gen_train = np.load("y_gender_train_utkface.npy")

X_test = np.load("X_test_utkface.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_utkface.npy")
y_gen_test = np.load("y_gender_test_utkface.npy")
valid_mask = (y_gen_train == 0) | (y_gen_train == 1)

X_train = X_train[valid_mask]
y_gen_train = y_gen_train[valid_mask]
y_age_train = y_age_train[valid_mask]

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AgeGenderCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_utk.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")
    

100%|██████████| 536/536 [00:10<00:00, 49.04it/s]


Epoch 1: Train Loss = 390.4465 | Val Loss = 355.7295
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:10<00:00, 49.29it/s]


Epoch 2: Train Loss = 308.5065 | Val Loss = 289.8488
Yeni en iyi model kaydedildi.


100%|██████████| 536/536 [00:25<00:00, 21.18it/s]


...

Epoch 28: Train Loss = 59.5055 | Val Loss = 301.5871


100%|██████████| 536/536 [00:45<00:00, 11.75it/s]


Epoch 29: Train Loss = 60.9210 | Val Loss = 257.9290


100%|██████████| 536/536 [00:45<00:00, 11.72it/s]


Epoch 30: Train Loss = 60.6094 | Val Loss = 196.0804


In [8]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN().to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.7753044939101218
Cinsiyet - Precision: 0.7676330592816962
Cinsiyet - Recall   : 0.7689640225400953
Cinsiyet - F1-score : 0.7682979644867908
Yaş - MAE           : 10.086148354822926
Yaş - RMSE          : 13.653783545645567


In [9]:
X_train = np.load("X_train_all_imdbwiki.npy", allow_pickle=True)
y_age_train = np.load("y_age_train_all_imdbwiki.npy")
y_gen_train = np.load("y_gender_train_all_imdbwiki.npy")

X_test = np.load("X_test_all_imdbwiki.npy", allow_pickle=True)
y_age_test = np.load("y_age_test_all_imdbwiki.npy")
y_gen_test = np.load("y_gender_test_all_imdbwiki.npy")

X_train_split, X_val_split, y_gen_train_split, y_gen_val_split, y_age_train_split, y_age_val_split = train_test_split(X_train, y_gen_train, y_age_train, test_size=0.1, shuffle=False)

train_dataset = AgeGenderDataset(X_train_split, y_gen_train_split, y_age_train_split)
val_dataset   = AgeGenderDataset(X_val_split, y_gen_val_split, y_age_val_split)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderCNN(input_size=(64, 64)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion_g = nn.BCELoss()
criterion_a = nn.MSELoss()

best_model_path = "best_model_imdbwiki.pth"
best_val_loss = float('inf')
for epoch in range(30):
    train_loss = train_model(model, train_loader, criterion_g, criterion_a, optimizer, device)
    val_loss   = evaluate_model(model, val_loader, criterion_g, criterion_a, device)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print("Yeni en iyi model kaydedildi.")

100%|██████████| 11459/11459 [03:13<00:00, 59.24it/s] 


Epoch 1: Train Loss = 184.9071 | Val Loss = 145.2506
Yeni en iyi model kaydedildi.


100%|██████████| 11459/11459 [01:57<00:00, 97.76it/s] 


Epoch 2: Train Loss = 165.1480 | Val Loss = 145.9840


100%|██████████| 11459/11459 [01:41<00:00, 112.62it/s]


...

Epoch 28: Train Loss = 91.9881 | Val Loss = 143.7075


100%|██████████| 11459/11459 [01:43<00:00, 111.11it/s]


Epoch 29: Train Loss = 90.7922 | Val Loss = 147.1880


100%|██████████| 11459/11459 [01:40<00:00, 114.04it/s]


Epoch 30: Train Loss = 89.8116 | Val Loss = 144.0882


In [10]:
test_dataset = AgeGenderDataset(X_test, y_gen_test, y_age_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
model = AgeGenderCNN(input_size=(64, 64)).to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()
# Modeli test et ve sonuçları yazdır
test_model(model, test_dataloader, device)


=== [TEST SONUÇLARI] ===
Cinsiyet - Accuracy : 0.7532670273242286
Cinsiyet - Precision: 0.7490699201225517
Cinsiyet - Recall   : 0.8897806661251015
Cinsiyet - F1-score : 0.8133846222393846
Yaş - MAE           : 8.932095008119227
Yaş - RMSE          : 11.79966534389093
